# Batch Scoring: New Clients

Automated scoring pipeline for new, unlabelled clients:

1. Picks up new client CSVs (`inference_*.csv`) dropped in the shared UC volume `/Volumes/workspace/default/churn_raw` — ingests them into the raw table (skipping clients already known) and records which file each client came from
   - Only clients that have **never been scored** are scored on each run
2. Applies **the same** feature engineering as training — reads `02_etl_and_features/sql/feature_store.sql` directly, so this notebook can never drift from the training pipeline
3. Scores with the **champion** model alias (`@champion` — promoting a new champion requires no edit here)
4. Applies the deploy cut-off (0.29, best F1) and **appends** the full row + scores to the output table with a `scored_at` timestamp and the `source_file` each client came from

Designed to run unattended in a Lakeflow Job. All five parameters below can be overridden per run by the job definition.

In [0]:
%pip install feature-engine lightgbm

In [0]:
import os
from datetime import datetime

# --- Job-overridable parameters ---
dbutils.widgets.text("source_table", "workspace.default.inference", "Raw clients table")
dbutils.widgets.text("model_uri", "models:/workspace.default.churn_lightgbm_balanced_tuned@champion", "Champion model URI")
dbutils.widgets.text("threshold", "0.29", "Deploy cut-off (best F1)")
dbutils.widgets.text("output_table", "workspace.default.churn_scores", "Scored output table (append)")
dbutils.widgets.text("ingest_folder", "/Volumes/workspace/default/churn_raw", "UC volume where new client CSVs (inference_*.csv) are dropped — shared across the team, watched by the job trigger")

SOURCE_TABLE = dbutils.widgets.get("source_table")
MODEL_URI    = dbutils.widgets.get("model_uri")
THRESHOLD    = float(dbutils.widgets.get("threshold"))
OUTPUT_TABLE = dbutils.widgets.get("output_table")
# Resolve the (possibly relative) path — so any override style works
INGEST_FOLDER = os.path.abspath(dbutils.widgets.get("ingest_folder"))

# Timestamp of this scoring round — separates rounds in the append-only output table
SCORED_AT = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

print(f"source   : {SOURCE_TABLE}")
print(f"model    : {MODEL_URI}")
print(f"threshold: {THRESHOLD}")
print(f"output   : {OUTPUT_TABLE} (append, round {SCORED_AT})")
print(f"watching : {INGEST_FOLDER}")

In [0]:
import fnmatch
import os
import pandas as pd
from pyspark.sql import functions as F

# --- 1. Discover new client files dropped in the ingest folder (naming convention: inference_*.csv) ---
assert os.path.isdir(INGEST_FOLDER), f"Ingest folder not found: {INGEST_FOLDER} — check the ingest_folder parameter"
new_files = sorted(
    f for f in os.listdir(INGEST_FOLDER) if fnmatch.fnmatch(f.lower(), "inference_*.csv")
)

print(f"New client files found: {new_files or 'none'}")

# --- 2. Read each file, align columns and types with the raw table, tag rows with the file name ---
raw_tbl = spark.table(SOURCE_TABLE)
raw_schema = raw_tbl.schema

raw_parts = []
for fname in new_files:
    df_csv = (spark.read
        .option("header", True)
        .option("multiLine", True)  # CustomerFeedback contains newlines inside quoted fields
        .csv(f"{INGEST_FOLDER}/{fname}"))
    missing = [c for c in raw_schema.fieldNames() if c not in df_csv.columns]
    assert not missing, f"{fname} is missing columns vs {SOURCE_TABLE}: {missing}"
    typed = df_csv.select([F.col(c).cast(raw_schema[c].dataType).alias(c) for c in raw_schema.fieldNames()])
    raw_parts.append(typed.withColumn("_source_file", F.lit(fname)))

# --- 3. Append only clients not yet in the raw table (dedup by customerID) ---
file_map_pd = pd.DataFrame(columns=["customerID", "source_file"])

if raw_parts:
    df_new_raw = raw_parts[0]
    for part in raw_parts[1:]:
        df_new_raw = df_new_raw.unionByName(part)

    df_new_clients = df_new_raw.join(raw_tbl.select("customerID"), "customerID", "left_anti")
    n_new = df_new_clients.count()

    if n_new > 0:
        (df_new_clients.drop("_source_file")
            .write.format("delta").mode("append").saveAsTable(SOURCE_TABLE))
        print(f"Ingested {n_new} new clients into {SOURCE_TABLE}")
    else:
        print("All clients in the new files already exist in the raw table — nothing ingested.")

    file_map_pd = (df_new_clients
        .select("customerID", F.col("_source_file").alias("source_file"))
        .toPandas())

# --- 4. Clients to score = in the raw table but never scored before ---
try:
    scored_ids = spark.table(OUTPUT_TABLE).select("customerID")
    clients_to_score = (spark.table(SOURCE_TABLE).select("customerID")
        .join(scored_ids, "customerID", "left_anti"))
except Exception:
    clients_to_score = spark.table(SOURCE_TABLE).select("customerID")  # output table doesn't exist yet

print(f"Clients to score this round: {clients_to_score.count()}")

In [0]:
# Run the EXACT same feature engineering used at training time.
# feature_store.sql contains a {dataset} placeholder in the FROM clause.
FEATURE_SQL_PATH = "../../02_etl_and_features/sql/feature_store.sql"

with open(FEATURE_SQL_PATH) as f:
    feature_query = f.read().format(dataset="inference")

df_features = spark.sql(feature_query).join(clients_to_score, "customerID", "left_semi")

n_clients = df_features.count()
if n_clients == 0:
    print("No new clients to score — nothing to do this round.")
    dbutils.notebook.exit("No new clients to score.")
print(f"New clients to score: {n_clients}")
display(df_features.limit(5))

In [0]:
import mlflow

model = mlflow.sklearn.load_model(MODEL_URI)

X = df_features.toPandas()

# Train/serve consistency: the model must see exactly the features it was trained on
missing = [c for c in model.feature_names_in_ if c not in X.columns]
assert not missing, f"Feature columns missing vs champion model: {missing}"
assert list(model.classes_) == [0, 1], model.classes_  # class 1 (churn) must be predict_proba column 1

X["ProbaChurn"] = model.predict_proba(X[model.feature_names_in_])[:, 1]
X["prediction"] = (X["ProbaChurn"] >= THRESHOLD).map({True: "Yes", False: "No"})
X["scored_at"] = SCORED_AT

X[["customerID", "ProbaChurn", "prediction"]].head(10)

In [0]:
# Source file each client came from; 'backfill' when the client has no file record
if not file_map_pd.empty:
    X = X.merge(file_map_pd, on="customerID", how="left")
X["source_file"] = X["source_file"].fillna("backfill")

# Full audit row: every engineered feature the model saw + the 4 scoring columns
output_cols = list(df_features.columns) + ["ProbaChurn", "prediction", "scored_at", "source_file"]

spark_df = spark.createDataFrame(X[output_cols])

(spark_df.write
   .format("delta")
   .option("mergeSchema", "true")  # churn_scores already exists without source_file — column is added safely
   .mode("append")                 # append-only: every scoring round is preserved
   .saveAsTable(OUTPUT_TABLE))

print(f"Appended {spark_df.count()} scored clients to {OUTPUT_TABLE} (round {SCORED_AT})")

In [0]:
round_df = spark.sql(f"SELECT * FROM {OUTPUT_TABLE} WHERE scored_at = '{SCORED_AT}'")

n_scored = round_df.count()
n_yes    = round_df.filter("prediction = 'Yes'").count()
rev_risk = round_df.filter("prediction = 'Yes'").agg({"MonthlyCharges": "sum"}).first()[0] or 0

assert n_scored == n_clients, f"Round row count mismatch: scored {n_scored} vs input {n_clients}"

print(f"Scored this round   : {n_scored}")
print(f"Predicted churners  : {n_yes} ({n_yes / n_scored:.1%})")
print(f"Monthly rev at risk : ${rev_risk:,.2f}")
print(f"\nClients by source file this round:")
display(round_df.groupBy("source_file").count())

print(f"\nTop 10 highest-risk new clients:")
display(round_df.select("customerID", "ProbaChurn", "prediction", "ContractTier", "tenureGroup", "ServiceTier", "MonthlyCharges").orderBy("ProbaChurn", ascending=False).limit(10))